# Clase 051 — Desafíos del ML: overfitting, underfitting, datos insuficientes

Diagnosticamos sub/overfitting mirando la brecha train-validation, visualizamos el
bias-variance tradeoff y usamos regularización (`Ridge`) como contramedida.

Requiere: `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, learning_curve, validation_curve
from sklearn.metrics import r2_score

np.random.seed(42)
print('setup ok')

## 1. Dataset no lineal con ruido

Generamos `y = sin(x) + ruido`. Un modelo lineal simple *underfittea*; un polinomio de grado
alto *overfittea*.

In [ ]:
rng = np.random.default_rng(42)
n = 80
X = np.sort(rng.uniform(-3, 3, n)).reshape(-1, 1)
y = np.sin(X).ravel() + rng.normal(0, 0.3, n)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
print('train', Xtr.shape, 'test', Xte.shape)

## 2. Overfitting: polinomio de grado 15

Train casi perfecto pero test malo (o negativo): el modelo memorizó el ruido.

In [ ]:
over = make_pipeline(PolynomialFeatures(15), StandardScaler(), LinearRegression())
over.fit(Xtr, ytr)
r2_tr_over = r2_score(ytr, over.predict(Xtr))
r2_te_over = r2_score(yte, over.predict(Xte))
print(f'grado 15  -> train R2 {r2_tr_over:.3f} | test R2 {r2_te_over:.3f}')
assert r2_tr_over - r2_te_over > 0.1, 'esperamos brecha train>>test (overfitting)'
print('brecha grande train-test = overfitting (baja bias, alta variance)')

## 3. Underfitting: grado 1 sobre datos no lineales

Ambos scores bajos y parecidos: el modelo es demasiado rígido para captar la curva.

In [ ]:
under = make_pipeline(PolynomialFeatures(1), LinearRegression())
under.fit(Xtr, ytr)
r2_tr_un = r2_score(ytr, under.predict(Xtr))
r2_te_un = r2_score(yte, under.predict(Xte))
print(f'grado 1   -> train R2 {r2_tr_un:.3f} | test R2 {r2_te_un:.3f}')
print('ambos bajos y parecidos = underfitting (alto bias)')

## 4. Regularización: Ridge sobre las mismas features polinomiales

`Ridge` penaliza la norma de los coeficientes: baja variance a cambio de algo de bias.
Debería mejorar el test respecto al grado 15 sin regularizar.

In [ ]:
reg = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=1.0))
reg.fit(Xtr, ytr)
r2_tr_reg = r2_score(ytr, reg.predict(Xtr))
r2_te_reg = r2_score(yte, reg.predict(Xte))
print(f'Ridge a=1 -> train R2 {r2_tr_reg:.3f} | test R2 {r2_te_reg:.3f}')
assert r2_te_reg > r2_te_over, 'Ridge deberia mejorar el test vs grado 15 sin regularizar'
print('la regularizacion recupera generalizacion')

## 5. Validation curve: score vs alpha (fuerza de regularización)

Barremos `alpha` para encontrar el sweet spot bias-variance.

In [ ]:
alphas = np.logspace(-3, 3, 12)
model = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge())
tr_sc, va_sc = validation_curve(model, X, y, param_name='ridge__alpha',
                                param_range=alphas, cv=5, scoring='r2')
best_alpha = alphas[va_sc.mean(axis=1).argmax()]
print(f'mejor alpha por CV: {best_alpha:.4g}  (R2 val {va_sc.mean(axis=1).max():.3f})')

## 6. Learning curve: ¿la brecha se cierra con más datos?

Si `val_score` sigue subiendo al agregar datos, faltan datos. Si ambas convergen bajo,
el cuello de botella es el modelo/las features.

In [ ]:
sizes, tr_lc, va_lc = learning_curve(
    make_pipeline(PolynomialFeatures(4), StandardScaler(), Ridge(alpha=1.0)),
    X, y, train_sizes=np.linspace(0.2, 1.0, 6), cv=5, scoring='r2')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogx(alphas, tr_sc.mean(axis=1), 'o-', label='train')
axes[0].semilogx(alphas, va_sc.mean(axis=1), 's-', label='validation')
axes[0].axvline(best_alpha, color='k', ls='--', lw=0.8)
axes[0].set_xlabel('alpha (Ridge)'); axes[0].set_ylabel('R2')
axes[0].set_title('Validation curve'); axes[0].legend()

axes[1].plot(sizes, tr_lc.mean(axis=1), 'o-', label='train')
axes[1].plot(sizes, va_lc.mean(axis=1), 's-', label='validation')
axes[1].set_xlabel('tamano de train'); axes[1].set_ylabel('R2')
axes[1].set_title('Learning curve'); axes[1].legend()
plt.tight_layout(); plt.show()

## Ejercicios

1. Repetí el diagnóstico con `make_regression(n_samples=50, noise=20)` y un polinomio de
   grado 15. ¿Es overfitting o underfitting? Justificá con los R2 de train y test.
2. Barré `Ridge(alpha)` en `[0.001, 0.01, 0.1, 1, 10, 100]` sobre el dataset del ejercicio 1
   y graficá la curva de validación. ¿Dónde está el sweet spot?
3. Simulá sampling bias: entrená con un train desbalanceado 90/10 (clasificación binaria) y
   testeá en un test balanceado. ¿Qué te oculta el accuracy global?
4. Compará `Lasso` vs `Ridge` sobre las features polinomiales. ¿Cuántos coeficientes lleva
   Lasso a 0 exacto? ¿Qué implica para selección de features?

## Conclusiones

- Overfitting = train alto, test bajo (baja bias, alta variance). Underfitting = ambos bajos.
- La regularización (`Ridge`/`Lasso`) cambia variance por algo de bias y suele recuperar test.
- La validation curve encuentra el `alpha` óptimo; la learning curve dice si faltan datos.
- El test set se separa al principio y no se usa para tunear (eso va por CV en el train).

## ✅ Soluciones de los ejercicios

Cada ejercicio del README resuelto y ejecutable. Corré las celdas de arriba antes para tener los imports en memoria; aun así cada solución vuelve a importar lo que necesita para poder leerse de forma autónoma.

**Ej. 1 — Diagnóstico visual.** Polinomio de grado 15 sobre datos casi lineales (`make_regression`) → *overfitting*. Y grado 1 sobre datos senoidales → *underfitting*.

In [ ]:

import numpy as np
from sklearn.datasets import make_regression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# (a) grado 15 sobre datos lineales con ruido
Xr, yr = make_regression(n_samples=50, n_features=1, noise=20.0, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(Xr, yr, test_size=0.3, random_state=42)
m15 = make_pipeline(PolynomialFeatures(15), StandardScaler(), LinearRegression()).fit(Xtr, ytr)
tr15, te15 = r2_score(ytr, m15.predict(Xtr)), r2_score(yte, m15.predict(Xte))
print(f"(a) grado 15: train R2={tr15:.3f} | test R2={te15:.3f} -> OVERFITTING (memoriza ruido)")
assert tr15 - te15 > 0.1, "esperamos brecha train>>test"

# (b) grado 1 sobre datos NO lineales (seno)
rng = np.random.default_rng(0)
Xs = np.sort(rng.uniform(-3, 3, 60)).reshape(-1, 1)
ys = np.sin(2 * Xs).ravel() + rng.normal(0, 0.2, 60)  # ~2 periodos: una recta no puede seguirlo
Xstr, Xste, ystr, yste = train_test_split(Xs, ys, test_size=0.3, random_state=42)
m1 = make_pipeline(PolynomialFeatures(1), LinearRegression()).fit(Xstr, ystr)
tr1, te1 = r2_score(ystr, m1.predict(Xstr)), r2_score(yste, m1.predict(Xste))
print(f"(b) grado 1 : train R2={tr1:.3f} | test R2={te1:.3f} -> UNDERFITTING (ambos bajos)")
assert tr1 < 0.4, "un modelo lineal no captura un seno oscilante: bias alto"

**Ej. 2 — Learning curve.** Train vs validación en función del tamaño de train. Si la brecha se cierra al agregar datos, conviene conseguir más; si convergen bajo, el modelo es demasiado simple.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import learning_curve

Xr, yr = make_regression(n_samples=400, n_features=1, noise=20.0, random_state=42)
est = make_pipeline(PolynomialFeatures(4), StandardScaler(), Ridge(alpha=1.0))
sizes, tr, va = learning_curve(est, Xr, yr, train_sizes=np.linspace(0.1, 1.0, 6),
                               cv=5, scoring="r2")
tr_m, va_m = tr.mean(1), va.mean(1)
gap = tr_m[-1] - va_m[-1]
print(f"R2 train final={tr_m[-1]:.3f}, val final={va_m[-1]:.3f}, brecha={gap:.3f}")
print("brecha chica y val alto -> el modelo generaliza; agregar datos rinde poco extra")
plt.plot(sizes, tr_m, "o-", label="train"); plt.plot(sizes, va_m, "s-", label="validation")
plt.xlabel("tamano de train"); plt.ylabel("R2"); plt.legend(); plt.title("Learning curve"); plt.show()
assert va_m[-1] > va_m[0], "mas datos no deberia empeorar la validacion"

**Ej. 3 — Ridge vs alpha (curva de validación).** Barremos `alpha` y buscamos el sweet spot: ni demasiado chico (overfit) ni demasiado grande (underfit).

In [ ]:

import numpy as np
from sklearn.datasets import make_regression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import validation_curve

Xr, yr = make_regression(n_samples=80, n_features=1, noise=20.0, random_state=42)
alphas = np.array([0.001, 0.01, 0.1, 1, 10, 100])
model = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge())
tr, va = validation_curve(model, Xr, yr, param_name="ridge__alpha",
                          param_range=alphas, cv=5, scoring="r2")
best = alphas[va.mean(1).argmax()]
for a, vm in zip(alphas, va.mean(1)):
    print(f"alpha={a:8.3f} -> R2 val={vm:.3f}")
print(f"sweet spot: alpha={best} (R2 val maximo)")
assert 0.001 <= best <= 100

**Ej. 4 — Sampling bias.** Entrenamos con train desbalanceado 90/10 y testeamos en un test balanceado. El accuracy global engaña; hay que mirar recall por clase.

In [ ]:

import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

X, y = make_classification(n_samples=10000, n_classes=2, weights=[0.5, 0.5],
                           n_informative=5, random_state=42)
rng = np.random.default_rng(42)
# test balanceado
idx0, idx1 = np.where(y == 0)[0], np.where(y == 1)[0]
test_idx = np.concatenate([rng.choice(idx0, 1000, replace=False),
                           rng.choice(idx1, 1000, replace=False)])
mask = np.ones(len(y), bool); mask[test_idx] = False
Xpool, ypool = X[mask], y[mask]
p0, p1 = np.where(ypool == 0)[0], np.where(ypool == 1)[0]
# train SESGADO 90% clase0 / 10% clase1
train_idx = np.concatenate([rng.choice(p0, 900, replace=False),
                            rng.choice(p1, 100, replace=False)])
clf = LogisticRegression(max_iter=1000).fit(Xpool[train_idx], ypool[train_idx])
yp = clf.predict(X[test_idx])
acc = accuracy_score(y[test_idx], yp)
rec1 = recall_score(y[test_idx], yp, pos_label=1)
rec0 = recall_score(y[test_idx], yp, pos_label=0)
print(f"accuracy global={acc:.3f} | recall clase0={rec0:.3f} | recall clase1={rec1:.3f}")
print("el accuracy global oculta que la clase minoritaria en train se detecta peor")
assert rec0 - rec1 > -1  # siempre cierto; el punto es comparar recalls

**Ej. 5 — Feature engineering manual.** Predecir `monto` con estacionalidad semanal usando (a) solo el timestamp crudo vs (b) features de calendario (`dia_semana`, `mes`, `es_finde`). La diferencia de R² es el valor del feature engineering.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(42)
fechas = pd.date_range("2022-01-01", periods=730, freq="D")
dow = fechas.dayofweek.values
# monto con patron: mas alto los findes + estacionalidad mensual + ruido
monto = (100 + 40 * (dow >= 5) + 10 * np.sin(2 * np.pi * fechas.month.values / 12)
         + rng.normal(0, 8, len(fechas)))

ts = fechas.astype("int64").values.reshape(-1, 1)  # (a) timestamp crudo
Xb = np.column_stack([dow, fechas.month.values, (dow >= 5).astype(int)])  # (b)

r2_a = cross_val_score(RandomForestRegressor(n_estimators=80, random_state=42, n_jobs=1),
                       ts, monto, cv=5, scoring="r2").mean()
r2_b = cross_val_score(RandomForestRegressor(n_estimators=80, random_state=42, n_jobs=1),
                       Xb, monto, cv=5, scoring="r2").mean()
print(f"(a) solo timestamp : R2={r2_a:.3f}")
print(f"(b) calendario     : R2={r2_b:.3f}")
print(f"ganancia por feature engineering: {r2_b - r2_a:+.3f} de R2")
assert r2_b > r2_a, "las features de calendario deben capturar mejor la estacionalidad"